In [1]:
# !uv pip install ipykernel

In [2]:
# !uv pip install docling langchain-docling tabulate
# !uv python -m pip install rapidocr onnxruntime

In [3]:
import json
from pathlib import Path
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableStructureOptions,
    RapidOcrOptions,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)
from docling_core.types.doc import (
    ImageRefMode,
    PictureItem,
    TableItem,
)

f:\Full_Stack_GenerativeAI_And_Agentic_AI\gfs\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
PDF_PATH = Path(
    r"F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\complex_rag_parsing_sample_with_sunny_image.pdf"
)

OUTPUT_DIR = (
    Path(r"F:\Full_Stack_GenerativeAI_And_Agentic_AI")
    / "RAG"
    / "Data_parsing"
    / "data"
    / "docling_parsed_output"
)

PAGE_IMAGE_DIR = OUTPUT_DIR / "page_images"
PICTURE_DIR = OUTPUT_DIR / "extracted_pictures"
TABLE_DIR = OUTPUT_DIR / "extracted_tables"


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PICTURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

In [5]:
pipeline_options = PdfPipelineOptions()

# Enable OCR
pipeline_options.do_ocr = True

pipeline_options.ocr_options = RapidOcrOptions(lang=["en"])

# Enable table structure extraction
pipeline_options.do_table_structure = True

pipeline_options.table_structure_options = TableStructureOptions(
    do_cell_matching=True,
)

# Preserve page and picture images
pipeline_options.images_scale = 2.0
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

In [6]:
print("Parsing PDF with Docling...")

conversion_result = converter.convert(PDF_PATH)

Parsing PDF with Docling...


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1556.23it/s]
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
RapidOCR returned empty result!


2026-11-06 15:55:17,506 MatchingPostProcessor WARNING  2 of 35 pdf cells matched neither a row nor a column band of the 3x3 grid and were dropped from the table


In [7]:
docling_document = conversion_result.document

print("PDF parsed successfully.")
print("Total pages:", len(docling_document.pages))
print("Total tables:", len(docling_document.tables))
print("Total pictures:", len(docling_document.pictures))

PDF parsed successfully.
Total pages: 24
Total tables: 23
Total pictures: 10


In [8]:
json_output_path = OUTPUT_DIR / "docling_document.json"
json_output_path.write_text(
    json.dumps(
        docling_document.export_to_dict(),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
print("JSON saved:", json_output_path)

JSON saved: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\docling_parsed_output\docling_document.json


In [9]:
markdown_output_path = OUTPUT_DIR / "rag_ready_document.md"
docling_document.save_as_markdown(
    markdown_output_path,
    image_mode=ImageRefMode.REFERENCED,
)

print("Markdown saved:", markdown_output_path)

Markdown saved: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\docling_parsed_output\rag_ready_document.md


In [10]:
text_output_path = OUTPUT_DIR / "extracted_text.txt"

text_output_path.write_text(
    docling_document.export_to_markdown(strict_text=True),
    encoding="utf-8",
)

print("Plain text saved:", text_output_path)

Parameter `strict_text` has been deprecated and will be ignored.


Plain text saved: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\docling_parsed_output\extracted_text.txt


In [11]:
for page_number, page in docling_document.pages.items():

    if page.image is None:
        continue

    page_image_path = (
        PAGE_IMAGE_DIR
        / f"page_{page.page_no:03d}.png"
    )

    page.image.pil_image.save(
        page_image_path,
        format="PNG",
    )

print("Page images saved:", PAGE_IMAGE_DIR)

Page images saved: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\docling_parsed_output\page_images


In [12]:
picture_counter = 0

for element, _level in docling_document.iterate_items():

    if not isinstance(element, PictureItem):
        continue

    picture_counter += 1

    picture_path = (
        PICTURE_DIR
        / f"picture_{picture_counter:03d}.png"
    )

    picture_image = element.get_image(docling_document)

    if picture_image is not None:
        picture_image.save(
            picture_path,
            format="PNG",
        )

print("Pictures saved:", PICTURE_DIR)

Pictures saved: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\docling_parsed_output\extracted_pictures
